# Recreate `fid_Sigma_0923.mp4`

Renders a z-projected gas surface density movie (with velocity vectors and `nbody` star particles) from the SNeRHD FLASH run, matching `owzb_hundredth_Zsun_zhat_dens.mp4`.

Data: `research/data/sims/SNeRHD/RHDTurb_hdf5_plt_cnt_*` + `particle_files/RHDTurb_hdf5_part_*`.

In [1]:
import gc
import os
import time

import yt
from yt.utilities.exceptions import YTFieldNotFound

yt.set_log_level("warning")

In [2]:
DATA_DIR = "/Users/thummimm/Documents/research/data/sims/SNeRHD"
PART_DIR = os.path.join(DATA_DIR, "particle_files")
OUT_DIR = "/Users/thummimm/Documents/research/cca/outputs/fid_Sigma_frames"
os.makedirs(OUT_DIR, exist_ok=True)

field = ("gas", "density")
N_FRAMES = 200

In [6]:
t0 = time.time()

In [4]:
def make_frames(i):
    idx = f"{i:04d}"
    out_path = os.path.join(OUT_DIR, f"frame_{idx}.png")
    if os.path.exists(out_path):
        return #continue

    plt_fn = os.path.join(DATA_DIR, f"RHDTurb_hdf5_plt_cnt_{idx}")
    part_fn = os.path.join(PART_DIR, f"RHDTurb_hdf5_part_{idx}")
    ds = yt.load(plt_fn, particle_filename=part_fn)

    p = yt.ProjectionPlot(ds, "z", field, width=(2.8, "kpc"))
    p.set_unit(field, "Msun/pc**2")
    p.set_zlim(field, 1e1, 1e5)
    p.set_cmap(field, "magma")
    p.set_axes_unit("kpc")
    p.set_colorbar_label(field, r"Projected Dens $\left(\rm M_{\odot}/pc^{2}\right)$")
    p.set_figure_size(8)
    p.set_buff_size(1024)
    p.annotate_velocity(
        factor=16,
        color="k",
        headwidth=2,
        headlength=3,
        headaxislength=2.5,
        width=0.0012,
    )

    has_particles = False
    try:
        n = ds.all_data()["nbody", "particle_position_x"].size
        has_particles = n > 0
    except YTFieldNotFound:
        has_particles = False
    if has_particles:
        p.annotate_particles(
            (2.8, "kpc"), p_size=80, col="white", marker="*", ptype="nbody"
        )

    p.annotate_timestamp(corner="upper_left", text_args={"color": "black"})
    p.annotate_scale(corner="upper_right", size_bar_args={"color": "white"})
    p.save(out_path, mpl_kwargs={"dpi": 300})

    del ds, p
    gc.collect()

    elapsed = time.time() - t0
    
    print(f"frame {idx} done ({i + 1}/{N_FRAMES}), elapsed {elapsed:.1f}s", flush=True)


In [7]:
[make_frames(i) for i in range(N_FRAMES)]
print("ALL FRAMES DONE")

frame 0001 done (2/200), elapsed 178.1s
frame 0002 done (3/200), elapsed 180.8s
frame 0003 done (4/200), elapsed 183.4s
frame 0004 done (5/200), elapsed 186.0s
frame 0005 done (6/200), elapsed 188.6s
frame 0006 done (7/200), elapsed 192.3s
frame 0007 done (8/200), elapsed 195.6s
frame 0008 done (9/200), elapsed 198.7s
frame 0009 done (10/200), elapsed 201.9s
frame 0010 done (11/200), elapsed 204.9s
frame 0011 done (12/200), elapsed 208.1s
frame 0012 done (13/200), elapsed 211.1s
frame 0013 done (14/200), elapsed 214.2s
frame 0014 done (15/200), elapsed 217.3s
frame 0015 done (16/200), elapsed 220.5s
frame 0016 done (17/200), elapsed 223.7s
frame 0017 done (18/200), elapsed 227.6s
frame 0018 done (19/200), elapsed 230.6s
frame 0019 done (20/200), elapsed 233.8s
frame 0020 done (21/200), elapsed 236.8s
frame 0021 done (22/200), elapsed 239.9s
frame 0022 done (23/200), elapsed 243.1s
frame 0023 done (24/200), elapsed 246.6s
frame 0024 done (25/200), elapsed 249.7s
frame 0025 done (26/200)

KeyboardInterrupt: 

In [ ]:
for i in range(N_FRAMES):
    idx = f"{i:04d}"
    out_path = os.path.join(OUT_DIR, f"frame_{idx}.png")
    if os.path.exists(out_path):
        continue

    plt_fn = os.path.join(DATA_DIR, f"RHDTurb_hdf5_plt_cnt_{idx}")
    part_fn = os.path.join(PART_DIR, f"RHDTurb_hdf5_part_{idx}")
    ds = yt.load(plt_fn, particle_filename=part_fn)

    p = yt.ProjectionPlot(ds, "z", field, width=(2.8, "kpc"))
    p.set_unit(field, "Msun/pc**2")
    p.set_zlim(field, 1e1, 1e5)
    p.set_cmap(field, "magma")
    p.set_axes_unit("kpc")
    p.set_colorbar_label(field, r"Projected Dens $\left(\rm M_{\odot}/pc^{2}\right)$")
    p.set_figure_size(8)
    p.set_buff_size(1024)
    p.annotate_velocity(
        factor=16,
        color="k",
        headwidth=2,
        headlength=3,
        headaxislength=2.5,
        width=0.0012,
    )

    has_particles = False
    try:
        n = ds.all_data()["nbody", "particle_position_x"].size
        has_particles = n > 0
    except YTFieldNotFound:
        has_particles = False
    if has_particles:
        p.annotate_particles(
            (2.8, "kpc"), p_size=80, col="white", marker="*", ptype="nbody"
        )

    p.annotate_timestamp(corner="upper_left", text_args={"color": "black"})
    p.annotate_scale(corner="upper_right", size_bar_args={"color": "white"})
    p.save(out_path, mpl_kwargs={"dpi": 300})

    del ds, p
    gc.collect()

    elapsed = time.time() - t0
    print(f"frame {idx} done ({i + 1}/{N_FRAMES}), elapsed {elapsed:.1f}s", flush=True)



## Encode frames into the final movie

libx264 requires even width/height, hence the `scale` filter.

In [ ]:
movie_out = "/Users/thummimm/Documents/research/cca/outputs/fid_Sigma_0923.mp4"

!ffmpeg -y -framerate 30 -i {OUT_DIR}/frame_%04d.png \
    -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" \
    -c:v libx264 -pix_fmt yuv420p -crf 18 {movie_out}